# quantum-solar demo

Residential battery charge/discharge scheduling under time-of-use pricing,
solved as a QUBO with QAOA and verified against exact classical baselines.

The pipeline: `BatteryProblem` → `build_qubo` (exact slack encoding of the SoC
bounds) → `qubo_to_ising` → `QAOASolver`, cross-checked with `brute_force_solve`
(exact, tiny instances) and `dp_solve` (exact, scales to a full day).

In [ ]:
import numpy as np

from quantum_solar import (
    synthetic_instance, build_qubo, default_weights,
    brute_force_solve, dp_solve, QAOASolver,
)
from quantum_solar.plotting import plot_schedule

%matplotlib inline

## 1. A small instance

We use a 3-slot instance so the QUBO is small enough to enumerate exactly and to
run on the Aer simulator quickly.

In [ ]:
problem = synthetic_instance(num_slots=3, seed=7, capacity=3.0,
                             charge_energy=1.0, initial_soc=1.0)
weights = default_weights(problem)
qubo = build_qubo(problem, weights)

print(f"slots        : {problem.num_slots}")
print(f"QUBO vars    : {qubo.num_vars}  (2T decision bits + slack)")
print(f"prices       : {np.round(problem.price, 3)}")

## 2. Exact baselines

`brute_force_solve` enumerates the QUBO; `dp_solve` optimizes the true problem
with dynamic programming. They must agree on the optimal cost.

In [ ]:
brute = brute_force_solve(problem, qubo)
dp = dp_solve(problem)

print(f"brute force : cost ${brute.true_energy:.3f}  feasible={brute.feasible}")
print(f"DP          : cost ${dp.true_energy:.3f}  feasible={dp.feasible}")
assert np.isclose(brute.true_energy, dp.true_energy)

## 3. QAOA on the Aer simulator

QAOA should recover the same optimum the exact solvers found.

In [ ]:
result = QAOASolver(reps=3, n_starts=5, shots=4096, seed=1234).solve(problem, qubo)

print(f"QAOA        : cost ${result.true_energy:.3f}  feasible={result.feasible}")
c, d = problem.decode(result.x)
print(f"charge bits : {c}")
print(f"discharge   : {d}")
assert np.isclose(result.true_energy, dp.true_energy)

## 4. A full-day schedule

The DP solver scales, so we can plan an entire day and visualize the schedule.

In [ ]:
day = synthetic_instance(num_slots=12, seed=1, noise=0.0)
day_solution = dp_solve(day)
print(f"daily cost  : ${day_solution.true_energy:.2f}")

fig = plot_schedule(day, day_solution)